# BayesFlow on the same test datasets as TINN and PyDDM (all five models), Google Colab

**Before running**
1. Runtime > Change runtime type > **GPU**.
2. Zip the `results` folder made by `comparison_local.ipynb` (`results.zip`). For every model it must hold
   `test_datasets.npz` (and `tinn.csv`, `pyddm.csv` for the comparison figures).

**Steps**
1. Install packages (the first time the session restarts itself; then Run all again).
2. Get this repository and `results.zip` (upload, or from Google Drive).
3. Train BayesFlow once per model in `MODELS_TO_RUN` (amortized: simulated training data from the prior =
   the test-dataset ranges, same simulator, same number of trials) and infer the posterior for every test dataset.
4. Rebuild the comparison tables and figures with all three methods.
5. Download `results_with_bayesflow.zip` and put it in place of the local `results` folder.

## 1. Install

In [ ]:
import importlib.util, os
if importlib.util.find_spec("pyddm") is None or importlib.util.find_spec("bayesflow") is None:
    !pip install -q pyddm bayesflow numba
    print("installed; restarting the session - run all cells again")
    os.kill(os.getpid(), 9)
import torch
print("packages ready; GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none (Runtime > Change runtime type > GPU)")

## 2. Code (this repository)

In [ ]:
REPO_URL = "https://github.com/NMohammadi76/TINN-framework.git"   # this repository
import os
if not os.path.exists("/content/TINN"):
    !git clone -q {REPO_URL} /content/TINN
CMP = "/content/TINN/2_parameter_estimation"
print("code:", CMP)

## 3. TINN and PyDDM results

Upload `results.zip`, or set `USE_DRIVE = True` and give its path in Google Drive.

In [ ]:
USE_DRIVE = False
DRIVE_ZIP = "/content/drive/MyDrive/results.zip"

import shutil, zipfile
os.chdir("/content")
if not os.path.exists("/content/results"):
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        zip_path = DRIVE_ZIP
    else:
        from google.colab import files
        zip_path = next(iter(files.upload()))
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content/_unzip")
    # accept both results.zip containing results/ and a zip of its contents
    found = [r for r, ds, fs in os.walk("/content/_unzip") if "test_datasets.npz" in fs]
    root = os.path.dirname(found[0]) if found else None
    assert root, "no test_datasets.npz found in the zip"
    shutil.move(root, "/content/results")
for m in sorted(os.listdir("/content/results")):
    p = os.path.join("/content/results", m)
    if os.path.isdir(p):
        print(m, sorted(f for f in os.listdir(p) if f.endswith((".npz", ".csv")) and "part" not in f))

## 4. Settings

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"     # "torch" (GPU on Windows with the CUDA build) or "jax"

MODELS_TO_RUN = ["ddm", "urgency", "ou", "dmc", "collapse"]   # each needs results/<model>/test_datasets.npz
N_SIM = 20000            # training datasets per model (from the prior, same n_trials as the test datasets)
N_VAL = 300              # validation datasets
EPOCHS = 100
BATCH_SIZE = 128
SUMMARY_DIM = 64
INFERENCE_NET = "diffusion"   # "diffusion" (as in your code) or "coupling" (faster to sample)
N_SAMPLES = 1000         # posterior draws per test dataset
SEED = 7

## 5. Shared code

In [ ]:
import sys, time, subprocess
import numpy as np, pandas as pd

sys.path.insert(0, CMP)
from common import load_test_datasets, sample_params, simulate_batch, params_of, ranges_of, MODELS, T_MAX
RESULTS = "/content/results"

import bayesflow as bf

LABELS = {"v": "Drift rate", "a": "Boundary", "t0": "Non-decision time", "v0": "Drift rate v0", "v1": "Urgency v1",
          "tau": "Time constant", "beta": "Leak", "mu0": "Drift rate mu0", "A": "Automatic amplitude",
          "T_inf": "Collapse time"}


def to_features(model, rt, ch, cond):
    """log RT, choice (+1 upper / -1 lower) and, for DMC, the condition (+1 / -1) of every trial."""
    rt = np.where(np.isnan(rt), T_MAX, rt)
    feats = [np.log(rt), 2.0 * ch - 1.0]
    if len(MODELS[model]["conditions"]) > 1:
        feats.append(np.asarray(MODELS[model]["conditions"])[cond.astype(int)])
    return np.stack(feats, -1).astype("float32")


def training_data(model, n_sim, n_trials, seed):
    p = sample_params(model, np.random.default_rng(seed), n_sim)   # prior = test-dataset ranges
    rt, ch, cond = simulate_batch(model, p, n_trials, seed=seed)
    ch = np.where(np.isnan(rt), 0.5, ch)
    out = {k: p[k][:, None].astype("float32") for k in params_of(model)}
    out["sim_data"] = to_features(model, rt, ch, cond)
    return out


def make_workflow(model):
    params, ranges = params_of(model), ranges_of(model)
    adapter = bf.adapters.Adapter().convert_dtype("float64", "float32")
    for k in params:                                   # posterior stays inside the prior range
        adapter = adapter.constrain(k, lower=ranges[k][0], upper=ranges[k][1])
    adapter = adapter.concatenate(params, into="inference_variables").rename("sim_data", "summary_variables")
    inference_net = bf.networks.DiffusionModel() if INFERENCE_NET == "diffusion" else bf.networks.CouplingFlow()
    return bf.BasicWorkflow(adapter=adapter, summary_network=bf.networks.DeepSet(summary_dim=SUMMARY_DIM),
                            inference_network=inference_net, standardize="inference_variables")


def run_script(*args):
    """make_figures / make_summary_table in their own process (BayesFlow switched torch autograd off here)."""
    p = subprocess.run([sys.executable, os.path.join(CMP, args[0]), *args[1:]], cwd=CMP,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print("\n".join(l for l in p.stdout.splitlines() if "Warning" not in l)[-3000:])


def fit_model(model):
    R = os.path.join(RESULTS, model)
    d = load_test_datasets(os.path.join(R, "test_datasets.npz"))
    params, n_trials = params_of(model), d["rt"].shape[1]
    print(f"===== {model}: {params}, prior {ranges_of(model)}, {len(d['rt'])} test datasets x {n_trials} trials")

    # ---- simulate training data and train (once per model)
    t = time.time()
    train = training_data(model, N_SIM, n_trials, SEED)
    val = training_data(model, N_VAL, n_trials, SEED + 1)
    t_sim = time.time() - t
    workflow = make_workflow(model)
    t = time.time()
    workflow.fit_offline(train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=val, verbose=2)
    t_train = time.time() - t
    print(f"simulation {t_sim:.0f}s, training {t_train / 60:.1f} min")

    # ---- posterior on the same test datasets as TINN and PyDDM
    x = to_features(model, d["rt"], d["choice"], d["cond"])
    t = time.time()
    post = workflow.sample(conditions={"sim_data": x}, num_samples=N_SAMPLES)
    t_inf = time.time() - t
    rows = []
    for i in range(len(x)):
        row = {"dataset": i, "time_s": t_inf / len(x), "train_sim_s": t_sim, "train_fit_s": t_train}
        for k in params:
            s = np.asarray(post[k][i]).ravel()
            row[f"{k}_est"] = float(s.mean())                 # point estimate = posterior mean
            row[f"{k}_median"] = float(np.median(s))
            row[f"{k}_q025"], row[f"{k}_q975"] = map(float, np.quantile(s, [0.025, 0.975]))
            row[f"{k}_true"] = float(d[f"{k}_true"][i])
        rows.append(row)
    df = pd.DataFrame(rows)

    # ---- BayesFlow's own diagnostics (appendix): recovery on the test datasets, calibration on validation data
    os.makedirs(os.path.join(R, "paper"), exist_ok=True)
    targets = {k: np.asarray(d[f"{k}_true"], "float32")[:, None] for k in params}
    f = bf.diagnostics.recovery(post, targets, variable_keys=params, variable_names=[LABELS[k] for k in params],
                                xlabel="True value", ylabel="Posterior estimate", color="black",
                                title_fontsize=14, label_fontsize=12, figsize=(4 * len(params), 4))
    f.savefig(os.path.join(R, "paper", "bayesflow_recovery_test.pdf"), bbox_inches="tight")
    for name, fig in workflow.plot_default_diagnostics(test_data=val, num_samples=300).items():
        fig.savefig(os.path.join(R, "paper", f"bayesflow_{name}.pdf"), bbox_inches="tight")

    df.to_csv(os.path.join(R, "bayesflow.csv"), index=False)     # written last: a finished model is skipped later
    cover = {k: round(float(((df[f"{k}_q025"] <= df[f"{k}_true"]) & (df[f"{k}_true"] <= df[f"{k}_q975"])).mean()), 2)
             for k in params}
    print(f"inference {t_inf:.1f}s for {len(x)} datasets; 95% coverage {cover}")

    # ---- comparison figures and tables, now with all three methods (same format as before)
    run_script("make_figures.py", "--results", R)

## 6. Run for every model

In [ ]:
for model in MODELS_TO_RUN:
    R = os.path.join(RESULTS, model)
    if not os.path.exists(os.path.join(R, "test_datasets.npz")):
        print(f"{model}: no test datasets yet, run comparison_local.ipynb for it first - skipped")
    elif os.path.exists(os.path.join(R, "bayesflow.csv")):
        print(f"{model}: bayesflow.csv already exists - skipped (delete it to run again)")
    else:
        fit_model(model)

## 7. Summary table and figures

In [ ]:
run_script("make_summary_table.py", "--results", RESULTS)

from IPython.display import Image, display
for model in MODELS_TO_RUN:
    f = os.path.join(RESULTS, model, "paper", "fig1_recovery.png")
    if os.path.exists(f):
        print(model)
        display(Image(f, width=900))

## 8. Download the results

In [ ]:
from google.colab import files
files.download(shutil.make_archive("/content/results_with_bayesflow", "zip", "/content", "results"))